In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Install and Imports

In [ ]:
!pip install sentencepiece
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.4 MB/s eta 0:00:00
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 74.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.8/236.8 kB 26.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 101.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 85.1 MB/s eta 0:00:00


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration, T5Config, AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.optimization import AdamW
from tqdm import tqdm

## Model

In [ ]:
# Define the dataset class
class KeyTextDataset(Dataset):
    def __init__(self, keys, texts, tokenizer):
        self.keys = keys
        self.texts = texts
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]
        text = self.texts[idx]
        key_encoding = self.tokenizer(
            key,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
        )

        text_encoding = self.tokenizer(
            text,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
        )
        input_ids = key_encoding['input_ids'].squeeze()
        attention_mask = key_encoding['attention_mask'].squeeze()

        # print(text_encoding)

        labels = text_encoding['input_ids'].squeeze()
        labels[labels == 0] = -100
        labels_attention_mask = text_encoding['attention_mask'].squeeze()

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'labels_attention_mask':labels_attention_mask,
            'text': text
        }

# Function to train the model
def train_model(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0

    # progress_bar = tqdm(enumerate(dataloader), total=len(dataloader))
    for step, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        labels_attention_mask = batch['labels_attention_mask'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_attention_mask=labels_attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        # progress_bar.set_description(f"Train Loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

# Function to validate the model
def validate_model(model, dataloader, device):
    model.eval()
    total_loss = 0

    # progress_bar = tqdm(enumerate(dataloader), total=len(dataloader))
    for step, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        labels_attention_mask = batch['labels_attention_mask'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_attention_mask=labels_attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        # progress_bar.set_description(f"Train Loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

# Function to save the trained model and tokenizer
def save_model(model, tokenizer, output_dir):
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Model and tokenizer saved to '{output_dir}'")

# Function to load the saved model and tokenizer
def load_model(output_dir):
    model = AutoModelForSeq2SeqLM.from_pretrained(output_dir)
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    print(f"Model and tokenizer loaded from '{output_dir}'")
    return model, tokenizer

In [ ]:
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_dir = "/content/drive/MyDrive/BengaliKey2Text/ModelV10V2"
# Initialize the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)
model.to(device)

MT5ForConditionalGeneration(
  (shared): Embedding(250112, 768)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 768)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
         

## Dataset Load

In [ ]:
import pandas as pd
df =  pd.read_csv('/content/drive/MyDrive/BengaliKey2Text/DataV3/strkeysDfV01.csv')
df.columns = ["keywords", "text"]
# df = df.iloc[1579900:2000000]
# df = df.iloc[1579900:1679900]
df = df.iloc[1679900:2000000]
df = df.reset_index(drop=True)
# df = df.head(500)
df

,keywords,text
0,শুরু শেষে চারটায় চূড়ান্তপর্ব জমজমাট বিকেল লড়াই,দিনব্যাপী জমজমাট যুক্তির লড়াই শেষে বিকেল চারটা...
1,বলেন লাভজনক বিশ্ববিদ্যালয় নয় প্রতিষ্ঠান,"হাছান মাহমুদ বলেন, বিশ্ববিদ্যালয় কোনো লাভজনক প..."
2,এদের বিভাগীয় ঘটনার থেকে শিকারে যে যায় তদন্ত বি...,"সজল চৌধুরীকে অপহরণকারীদের মতো বিপজ্জনক, চৌকস ও..."
3,নিয়মিত গেলেও পড়তে কেন্দ্রে দেশবিদেশের ত্বকের চ...,নিয়মিত সৌন্দর্যচর্চা কেন্দ্রে না গেলেও বাসায় ত...
4,টেস্ট সঙ্গে ও জয় ভারত নিউজিল্যান্ড ওয়েস্ট ড্র ...,জিম্বাবুয়ে ও ওয়েস্ট ইন্ডিজের বিপক্ষে টেস্ট জয় ...
...,...,...
320095,চাইলে বলেন ধরনের জানতে নারীর নিয়ে শোনা কথা নান...,"ছবিটির গল্প সম্পর্কে জানতে চাইলে টয়া বলেন, ‘না..."
320096,কর্মী করতে কারখানায় হবে উৎপাদনের,আওয়ামী লীগকে কর্মী উৎপাদনের কারখানায় পরিণত করত...
320097,গেছে অভিমান ছুঁয়ে তাঁর,সে অভিমান ছুঁয়ে গেছে তাঁর মেয়েদেরও।
320098,কিছু বসে আলী বিক্রি হাটে গ্রামের কাঁঠালের মন্দ...,কিন্তু রোজার কারণে বাজার কিছু মন্দা।’ তিনি জান...


In [ ]:
print(df.isnull().sum())

keywords    0
text        0
dtype: int64


In [ ]:
df.dropna(inplace=True)

In [ ]:
print(df.isnull().sum())

keywords    0
text        0
dtype: int64


In [ ]:
# Load your dataset
keys = df['keywords'].tolist()  # List of keys
texts = df['text'].tolist()  # List of corresponding texts

In [ ]:
type(texts)

list

## Train

In [ ]:
# Split your dataset into train and validation sets
train_keys = keys[:int(len(keys)*0.8)]
train_texts = texts[:int(len(texts)*0.8)]
valid_keys = keys[int(len(keys)*0.8):]
valid_texts = texts[int(len(texts)*0.8):]

# Create the train and validation datasets
train_dataset = KeyTextDataset(train_keys, train_texts, tokenizer)
valid_dataset = KeyTextDataset(valid_keys, valid_texts, tokenizer)

# Create data loaders
batch_size = 8
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

# Set the training parameters
num_epochs = 2
learning_rate = 0.0001
warmup_steps = 500
total_steps = len(train_dataloader) * num_epochs

# Set the optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=learning_rate,
                                                total_steps=total_steps, div_factor=10,
                                                final_div_factor=100,
                                                pct_start=0.1,
                                                anneal_strategy='linear')

# Set the number of epochs for early stopping
patience = 3
best_valid_loss = float('inf')
epochs_no_improve = 0

# Start training
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    train_loss = train_model(model, tqdm(train_dataloader), optimizer, device)
    # print(f"Train Loss: {train_loss:.4f}")

    # Validate the model
    valid_loss = validate_model(model, tqdm(valid_dataloader), device)
    # print(f"Valid Loss: {valid_loss:.4f}")

    # Early stopping check
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve == patience:
            print("Early stopping triggered!")
            break

    # Adjust the learning rate
    scheduler.step()
    # Print progress
    print('\n')
    print(f'Epoch: {epoch + 1} \tTraining Loss: {train_loss:.6f} \tValidation Loss: {valid_loss:.6f}')
    print('\n\n')

In [ ]:
# Set the output directory for saving the model
output_dir = "/content/drive/MyDrive/BengaliKey2Text/ModelV10V3"
# output_dir = "/content/drive/MyDrive/BengaliKey2Text/ModelVtest500"

# Save the trained model and tokenizer
save_model(model, tokenizer, output_dir)

Model and tokenizer saved to '/content/drive/MyDrive/BengaliKey2Text/ModelV10V0.32M'


## Predictiion

In [ ]:
#loading_model_dir = "/content/drive/MyDrive/BengaliKey2Text/ModelV9"
loading_model_dir = "/content/drive/MyDrive/BengaliKey2Text/ModelV10V3"
# loading_model_dir = "/content/drive/MyDrive/BengaliKey2Text/ModelVtest500"
loaded_model, loaded_tokenizer = load_model(loading_model_dir)
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.to(device)

Model and tokenizer loaded from '/content/drive/MyDrive/BengaliKey2Text/ModelV10V3'


MT5ForConditionalGeneration(
  (shared): Embedding(250112, 768)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 768)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
         

In [ ]:
# Function to generate text given a key
def generate_text(key):
    input_ids = loaded_tokenizer.encode(key, return_tensors='pt',add_special_tokens=True).to(device)

    with torch.no_grad():
      outputs = loaded_model.generate(
          input_ids=input_ids,
          max_length =64,
          num_beams =2,
          early_stopping =True,
          num_return_sequences = 1,
          repetition_penalty= 2.5,
          length_penalty= 1.0)

    # print(outputs)

    preds = [loaded_tokenizer.decode(g,skip_special_tokens=True,clean_up_tokenization_spaces=True) for g in outputs]

    generated_text = preds[0]
    return generated_text

def predict(key):
  return generate_text(key)

In [ ]:
key = "বিজ্ঞানীদের সফলতা মহাবিশ্ব অর্জন গবেষণার বড়"
predict(key)

'মহাবিশ্ব গবেষণার ক্ষেত্রে বিজ্ঞানীদের বড় সফলতা অর্জন করতে হবে।'

In [ ]:
key = "সফলতা বড় গবেষণার মহাবিশ্ব অর্জন বিজ্ঞানীদের"
predict(key)

'মহাবিশ্ব গবেষণার ক্ষেত্রে বিজ্ঞানীদের সফলতা সবচেয়ে বড় অর্জন।'

In [ ]:
key = "সম্বল জমিটাই বৃদ্ধার জীবিকা নির্বাহের"
predict(key)

'বৃদ্ধার জীবিকা ও জমিটাই নির্বাহের সম্বল।'

In [ ]:
key = "নির্বাহের জীবিকা বৃদ্ধার সম্বল জমিটাই"
predict(key)

'বৃদ্ধার জীবিকা, জমিটাই তাঁর নির্বাহের সম্বল।'

In [ ]:
key = "ব্লগিং অনুপ্রেরণা অভ্যাস এর লেখালেখির আমার ছোটবেলার"
predict(key)

'আমার ছোটবেলার লেখালেখির অভ্যাস ব্লগিং, এর অনুপ্রেরণা দেয়।'

In [ ]:
key = "আমার ছোটবেলার লেখালেখির অভ্যাস ব্লগিং এর অনুপ্রেরণা"
predict(key)

'আমার ছোটবেলার লেখালেখির অভ্যাস ব্লগিং এর অনুপ্রেরণা হিসেবে কাজ করে।'

In [ ]:
key = "প্রজন্ম চর্চায় সাহিত্য তরুণ"
predict(key)

'তরুণ প্রজন্ম সাহিত্য চর্চায় ব্যস্ত।'

In [ ]:
key = "সম্বল নির্বাহের বৃদ্ধার জমিটাই জীবিকা"
predict(key)

'বৃদ্ধার জীবিকা ও জমিটাই নির্বাহের সম্বল।'

In [ ]:
key = "সম্বল নির্বাহের বৃদ্ধার জমিটাই জীবিকা"
predict(key)

In [ ]:
key = "কেমন ডাটাসেট সময় ভাই বানাতে"
predict(key)

'ভাই, ডাটাসেট বানাতে কেমন সময় লাগে?'

In [ ]:
key = "কেমন ডাটাসেট সময় বানাতে"
predict(key)

'ডাটাসেট বানাতে কেমন সময় লাগে?'

In [ ]:
key = "ঠিকমতো এই অভাবের তাই শিক্ষার্থীরা কারণে"
predict(key)

'তাই এই অভাবের কারণে শিক্ষার্থীরা ঠিকমতো পড়তে পারছে না।'

In [ ]:
key = "ঠিকমতো এই অভাবের তাই শিক্ষার্থীরা কারণে"
predict(key)

'তাই এই অভাবের কারণে শিক্ষার্থীরা ঠিকমতো পড়তে পারছে না।'